# WiSig — Module2 SR1: Class-Conditional Support-Region Diagnosis

## Purpose
This notebook replaces the earlier score-fusion sequential router with a **support-region diagnosis** design.

The new Module2 answers one question in three states:
- **stable-known**: the sample is well supported by one known class and sits in that class core region
- **known-drift**: the sample is still supported by one known class, but lies outside the core region and inside the class outer support region
- **unknown**: the sample is not sufficiently supported by any known class region, or it is too ambiguous / too weakly supported locally

## Design principles
1. **Do not treat knownness as a black-box fusion score.** The main object is a class-conditional support region, not a blended confidence scalar.
2. **Use class-conditional geometry.** Distances are computed with shrinkage-diagonal Mahalanobis metrics rather than a single Euclidean radius.
3. **Use three interpretable signals only.**
   - nearest class support distance
   - class-conditional margin to the second-nearest class
   - local support consistency from source-neighbor evidence
4. **Allow light sub-prototype structure when the source class geometry is clearly multi-modal.**
5. **Keep outputs paper-ready.** Per-fold thresholds, class geometry summaries, raw diagnostics, and routing metrics are all saved.

## What changed relative to older Module2 versions
- Removed the old `Sid` score fusion of embedding-Mahalanobis + energy + prototype margin.
- Removed the old `Sdom` drift score and sequential soft router.
- Replaced them with a single geometric diagnosis chain based on **support regions**.

## Main notebook outputs
- `class_geometry.json`: per-class support-region statistics
- `support_region_thresholds.json`: per-class core / outer / margin / local-support thresholds
- `support_region_scores.npz`: raw diagnosis scores and predictions
- `summary_by_protocol.json`: protocol-level paper summary

In [10]:
import os, json, time
import itertools
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.cluster import KMeans

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path, dataset_name):
    with open(dataset_path + dataset_name + '.pkl', 'rb') as f:
        dataset = pickle.load(f)
    return dataset

compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

# ---- subset sizes ----
TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4

# ---- protocol suite aligned with Module 3 ----
TX_SPLIT_REPEATS = 5

RX_PROTOCOL_LIBRARY = {
    "3-9": dict(source_rx_num=3, drift_rx_num=9),
    "6-6": dict(source_rx_num=6, drift_rx_num=6),
    "9-3": dict(source_rx_num=9, drift_rx_num=3),
}
TX_PROTOCOL_LIBRARY = {
    "2-4": dict(known_tx_num=2, unknown_tx_num=4),
    "3-3": dict(known_tx_num=3, unknown_tx_num=3),
    "4-2": dict(known_tx_num=4, unknown_tx_num=2),
}

SELECTED_RX_PROTOCOLS = ["3-9", "6-6", "9-3"]
SELECTED_TX_PROTOCOLS = ["2-4", "3-3", "4-2"]
SELECTED_PROTOCOL_COMBOS = None

# ---- protocol ----
TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

# ---- signals ----
Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

# ---- training ----
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3

# ---- dims ----
D_DIM = 32

# ---- Sid fusion ----
FUSION_LAM = 0.5
SID_MARGIN_LAM = 0.75

# ---- sequential router calibration ----
FRR_TARGET = 0.05
FALSE_DRIFT_TARGET = 0.05
SOFT_ROUTE_UNKNOWN_PRIOR = 1.0
SOFT_ROUTE_DRIFT_PRIOR = 1.0
MIN_TEMP = 0.15
KNOWN_GATE_QUANTILE = 0.05
KNOWN_GATE_PROB_MIN = 0.55
KNOWN_GATE_PROB_MAX = 0.80
KNOWN_GATE_TEMP = 0.06
UNKNOWN_RECLAIM_LAM = 1.0


# ---- SR1 support-region diagnosis ----
SUPPORT_KNN = 15
KNN_MEM_PER_CLASS_DIAG = 256
CORE_Q = 0.75
OUT_Q = 0.95
MARGIN_Q = 0.10
LOCAL_Q = 0.10
MIN_CLASS_CALIB = 16
COV_SHRINK = 0.25
COV_FLOOR = 1e-3
SUBPROTO_AUTO = True
SUBPROTO_MAX = 2
SUBPROTO_MIN = 80
SUBPROTO_IMPROVE = 0.12

# ---- outputs ----
ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"../owdc_results/results_wisig_module2_sr1_support_region/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED,
    TX_TOTAL_USE=TX_TOTAL_USE,
    RX_TOTAL_USE=RX_TOTAL_USE,
    DAY_TOTAL_USE=DAY_TOTAL_USE,
    TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    RX_PROTOCOL_LIBRARY=RX_PROTOCOL_LIBRARY,
    TX_PROTOCOL_LIBRARY=TX_PROTOCOL_LIBRARY,
    SELECTED_RX_PROTOCOLS=SELECTED_RX_PROTOCOLS,
    SELECTED_TX_PROTOCOLS=SELECTED_TX_PROTOCOLS,
    SELECTED_PROTOCOL_COMBOS=SELECTED_PROTOCOL_COMBOS,
    TRAIN_DATES=TRAIN_DATES,
    TEST_DATES_RX=TEST_DATES_RX,
    TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ,
    D_FROM=D_FROM,
    MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE,
    EPOCHS=EPOCHS,
    LR=LR,
    WEIGHT_DECAY=WEIGHT_DECAY,
    PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES,
    DROPOUT=DROPOUT,
    D_DIM=D_DIM,
    FUSION_LAM=FUSION_LAM,
    SID_MARGIN_LAM=SID_MARGIN_LAM,
    FRR_TARGET=FRR_TARGET,
    FALSE_DRIFT_TARGET=FALSE_DRIFT_TARGET,
    SOFT_ROUTE_UNKNOWN_PRIOR=SOFT_ROUTE_UNKNOWN_PRIOR,
    SOFT_ROUTE_DRIFT_PRIOR=SOFT_ROUTE_DRIFT_PRIOR,
    MIN_TEMP=MIN_TEMP,
    KNOWN_GATE_QUANTILE=KNOWN_GATE_QUANTILE,
    KNOWN_GATE_PROB_MIN=KNOWN_GATE_PROB_MIN,
    KNOWN_GATE_PROB_MAX=KNOWN_GATE_PROB_MAX,
    KNOWN_GATE_TEMP=KNOWN_GATE_TEMP,
    UNKNOWN_RECLAIM_LAM=UNKNOWN_RECLAIM_LAM,
    SUPPORT_KNN=SUPPORT_KNN,
    KNN_MEM_PER_CLASS_DIAG=KNN_MEM_PER_CLASS_DIAG,
    CORE_Q=CORE_Q,
    OUT_Q=OUT_Q,
    MARGIN_Q=MARGIN_Q,
    LOCAL_Q=LOCAL_Q,
    MIN_CLASS_CALIB=MIN_CLASS_CALIB,
    COV_SHRINK=COV_SHRINK,
    COV_FLOOR=COV_FLOOR,
    SUBPROTO_AUTO=SUBPROTO_AUTO,
    SUBPROTO_MAX=SUBPROTO_MAX,
    SUBPROTO_MIN=SUBPROTO_MIN,
    SUBPROTO_IMPROVE=SUBPROTO_IMPROVE,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("RUN_DIR:", RUN_DIR)



Device: cuda
RUN_DIR: ../owdc_results/results_wisig_module2_v3/run_20260318_223701


In [11]:
# =========================
# Data utilities
# =========================
def get_idx(lst, val): 
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x_256_2):
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X_256_2, d_dim=32):
    Xc = iq_to_complex(X_256_2)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

In [12]:
# =========================
# Protocol suite construction
# =========================
tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)

def build_unique_tx_splits(tx_use, known_tx_num, n_splits, seed=42):
    all_known = [tuple(c) for c in itertools.combinations(list(tx_use), known_tx_num)]
    if n_splits > len(all_known):
        raise ValueError(
            f"Requested TX_SPLIT_REPEATS={n_splits}, but only {len(all_known)} unique TX splits "
            f"exist for C({len(tx_use)},{known_tx_num})."
        )
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(all_known))
    tx_splits = []
    for local_id, idx in enumerate(order[:n_splits], start=1):
        known = list(all_known[idx])
        unknown = [tx for tx in tx_use if tx not in known]
        tx_splits.append({
            "split_id": local_id,
            "known": known,
            "unknown": unknown,
        })
    return tx_splits

def build_protocol_specs(
    tx_use, rx_use,
    selected_rx_protocols, selected_tx_protocols,
    rx_protocol_library, tx_protocol_library,
    tx_split_repeats, seed=42, explicit_protocol_combos=None
):
    if explicit_protocol_combos is None:
        combo_pairs = [(rx_tag, tx_tag) for rx_tag in selected_rx_protocols for tx_tag in selected_tx_protocols]
    else:
        combo_pairs = list(explicit_protocol_combos)

    protocol_specs = []
    for combo_idx, (rx_tag, tx_tag) in enumerate(combo_pairs, start=1):
        if rx_tag not in rx_protocol_library:
            raise KeyError(f"Unknown RX protocol tag: {rx_tag}")
        if tx_tag not in tx_protocol_library:
            raise KeyError(f"Unknown TX protocol tag: {tx_tag}")

        rx_cfg = dict(rx_protocol_library[rx_tag])
        tx_cfg = dict(tx_protocol_library[tx_tag])

        src_n = int(rx_cfg["source_rx_num"])
        drift_n = int(rx_cfg["drift_rx_num"])
        known_n = int(tx_cfg["known_tx_num"])
        unknown_n = int(tx_cfg["unknown_tx_num"])

        if src_n + drift_n != len(rx_use):
            raise ValueError(
                f"RX protocol {rx_tag} expects source+drift={src_n + drift_n}, "
                f"but len(RX_USE)={len(rx_use)}."
            )
        if known_n + unknown_n != len(tx_use):
            raise ValueError(
                f"TX protocol {tx_tag} expects known+unknown={known_n + unknown_n}, "
                f"but len(TX_USE)={len(tx_use)}."
            )

        rng_rx = np.random.RandomState(seed + 1000 * combo_idx + 17 * src_n + drift_n)
        source_rxs = list(rng_rx.choice(list(rx_use), size=src_n, replace=False))
        source_rxs.sort()
        drift_rx = [r for r in rx_use if r not in source_rxs]

        tx_splits = build_unique_tx_splits(
            tx_use=tx_use,
            known_tx_num=known_n,
            n_splits=tx_split_repeats,
            seed=seed + 5000 + 97 * combo_idx,
        )

        protocol_specs.append(dict(
            protocol_tag=f"RX{rx_tag}_TX{tx_tag}",
            rx_protocol=rx_tag,
            tx_protocol=tx_tag,
            source_rx_num=src_n,
            drift_rx_num=drift_n,
            known_tx_num=known_n,
            unknown_tx_num=unknown_n,
            source_rxs=source_rxs,
            drift_rx=drift_rx,
            tx_splits=tx_splits,
        ))
    return protocol_specs

PROTOCOL_SPECS = build_protocol_specs(
    TX_USE, RX_USE,
    SELECTED_RX_PROTOCOLS, SELECTED_TX_PROTOCOLS,
    RX_PROTOCOL_LIBRARY, TX_PROTOCOL_LIBRARY,
    tx_split_repeats=TX_SPLIT_REPEATS,
    seed=SEED + 777,
    explicit_protocol_combos=SELECTED_PROTOCOL_COMBOS,
)

with open(os.path.join(RUN_DIR, "protocol_specs.json"), "w", encoding="utf-8") as f:
    json.dump(PROTOCOL_SPECS, f, indent=2)

print("Selected protocol combinations:", len(PROTOCOL_SPECS))
for spec in PROTOCOL_SPECS:
    print(
        f"[{spec['protocol_tag']}] "
        f"SOURCE_RXS={spec['source_rxs']} | DRIFT_RX={spec['drift_rx']} | "
        f"TXsplits={len(spec['tx_splits'])}"
    )


TX_USE: ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX_USE: ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE_USE: ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
Selected protocol combinations: 9
[RX3-9_TX2-4] SOURCE_RXS=[np.str_('19-2'), np.str_('2-1'), np.str_('20-1')] | DRIFT_RX=['1-1', '1-19', '14-7', '18-2', '2-19', '3-19', '7-14', '7-7', '8-8'] | TXsplits=5
[RX3-9_TX3-3] SOURCE_RXS=[np.str_('2-1'), np.str_('2-19'), np.str_('20-1')] | DRIFT_RX=['1-1', '1-19', '14-7', '18-2', '19-2', '3-19', '7-14', '7-7', '8-8'] | TXsplits=5
[RX3-9_TX4-2] SOURCE_RXS=[np.str_('19-2'), np.str_('2-1'), np.str_('20-1')] | DRIFT_RX=['1-1', '1-19', '14-7', '18-2', '2-19', '3-19', '7-14', '7-7', '8-8'] | TXsplits=5
[RX6-6_TX2-4] SOURCE_RXS=[np.str_('1-1'), np.str_('14-7'), np.str_('2-1'), np.str_('3-19'), np.str_('7-14'), np.str_('7-7')] | DRIFT_RX=['1-19', '18-2', '19-2', '2-19', '20-1', '8-8'] | TXsplits=5
[RX6-6_TX3-3] SOURCE_RXS=[np.st

In [13]:
# =========================
# Model: ResNet18-1D
# =========================
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [14]:
# =========================
# Train / inference helpers
# =========================
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss / max(1, total), total_correct / max(1, total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all, 0), np.concatenate(Z_all, 0)

def roc_auc(y_true, score):
    y_true = np.asarray(y_true)
    score = np.asarray(score)
    if len(np.unique(y_true)) < 2:
        return float('nan')
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

def l2norm(x, axis=1, eps=1e-12):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        P[k] = ZN[y == k].mean(axis=0)
    return l2norm(P, axis=1)

def cosine_to_proto(Z, P):
    return l2norm(Z, axis=1) @ P.T

def sid_EmbMaha(Z, mu_z, var_z):
    dif = Z[:, None, :] - mu_z[None, :, :]
    dist = np.sum((dif * dif) / (var_z[None, :, :] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train == k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_Energy(logits):
    m = logits.max(axis=1, keepdims=True)
    return (m + np.log(np.exp(logits - m).sum(axis=1, keepdims=True) + 1e-12)).squeeze(1).astype(np.float32)

def sid_ProtoMargin(cosine_scores):
    cs = np.asarray(cosine_scores, dtype=np.float32)
    if cs.shape[1] == 1:
        return cs[:, 0].astype(np.float32)
    top2 = np.partition(cs, kth=cs.shape[1] - 2, axis=1)[:, -2:]
    return (top2[:, 1] - top2[:, 0]).astype(np.float32)

def zscore(x, ref):
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if std < 1e-12:
        std = 1.0
    return ((x - mu) / std).astype(np.float32)

def robust_scale(x, min_scale=MIN_TEMP):
    x = np.asarray(x, dtype=np.float32)
    q75, q25 = np.quantile(x, [0.75, 0.25])
    iqr = float(q75 - q25)
    std = float(np.std(x))
    scale = max(iqr / 1.349 if iqr > 1e-8 else 0.0, std, min_scale)
    return float(scale)

def sigmoid_np(x):
    x = np.clip(x, -40.0, 40.0)
    return 1.0 / (1.0 + np.exp(-x))



In [15]:
# =========================
# Support-region diagnosis (SR1)
# =========================
def normalize_rows_np(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float32)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return (X / np.clip(n, eps, None)).astype(np.float32)


def shrink_diag_var(Z, shrink=COV_SHRINK, floor=COV_FLOOR):
    Z = np.asarray(Z, dtype=np.float32)
    if len(Z) == 0:
        return np.full((Z.shape[1],), floor, dtype=np.float32)
    v = Z.var(axis=0).astype(np.float32)
    mean_v = float(np.mean(v)) if v.size > 0 else float(floor)
    v = (1.0 - shrink) * v + shrink * mean_v
    return np.maximum(v, floor).astype(np.float32)


def _fit_single_region(Zk):
    center = Zk.mean(axis=0).astype(np.float32)
    var = shrink_diag_var(Zk)
    return [{"center": center, "var": var, "count": int(len(Zk))}]


def fit_class_support_regions(Z_train, y_train, K,
                              subproto_auto=SUBPROTO_AUTO,
                              subproto_max=SUBPROTO_MAX,
                              subproto_min=SUBPROTO_MIN,
                              subproto_improve=SUBPROTO_IMPROVE):
    """Fit one or two support regions per class.

    This is intentionally conservative:
    - default is still a single support region
    - a 2-subprototype upgrade is only used when KMeans clearly reduces within-class distortion
    - full covariance is avoided on purpose; shrinkage-diagonal covariance is more stable for this notebook
    """
    model = {"classes": [], "K": int(K)}
    geometry = []
    for k in range(K):
        Zk = np.asarray(Z_train[y_train == k], dtype=np.float32)
        cls = {"regions": []}
        if len(Zk) < max(8, subproto_min) or (not subproto_auto) or subproto_max <= 1:
            cls["regions"] = _fit_single_region(Zk)
            used_sub = 1
            improve = 0.0
        else:
            inertia1 = float(np.sum((Zk - Zk.mean(axis=0, keepdims=True)) ** 2))
            km = KMeans(n_clusters=2, random_state=SEED, n_init=10)
            labels = km.fit_predict(Zk)
            counts = np.bincount(labels, minlength=2)
            inertia2 = float(km.inertia_)
            improve = float((inertia1 - inertia2) / max(inertia1, 1e-8))
            if np.min(counts) < subproto_min or improve < subproto_improve:
                cls["regions"] = _fit_single_region(Zk)
                used_sub = 1
            else:
                used_sub = 2
                for m in range(2):
                    Zkm = Zk[labels == m]
                    cls["regions"].append({
                        "center": Zkm.mean(axis=0).astype(np.float32),
                        "var": shrink_diag_var(Zkm),
                        "count": int(len(Zkm)),
                    })
        model["classes"].append(cls)
        if len(Zk) > 0:
            class_var = shrink_diag_var(Zk)
            anis = float(np.max(class_var) / max(np.min(class_var), 1e-6))
        else:
            anis = float("nan")
        geometry.append({
            "class_id": int(k),
            "n_train": int(len(Zk)),
            "n_regions": int(used_sub),
            "subproto_improve": float(improve),
            "anisotropy_ratio": anis,
        })
    model["geometry"] = geometry
    return model


def class_distance_matrix(Z, support_model):
    Z = np.asarray(Z, dtype=np.float32)
    N = len(Z)
    K = support_model["K"]
    dist = np.full((N, K), np.inf, dtype=np.float32)
    sub_idx = np.zeros((N, K), dtype=np.int64)
    for k, cls in enumerate(support_model["classes"]):
        best = np.full((N,), np.inf, dtype=np.float32)
        best_m = np.zeros((N,), dtype=np.int64)
        for m, reg in enumerate(cls["regions"]):
            dif = Z - reg["center"].reshape(1, -1)
            d = np.sum((dif * dif) / np.maximum(reg["var"].reshape(1, -1), 1e-6), axis=1).astype(np.float32)
            upd = d < best
            best[upd] = d[upd]
            best_m[upd] = int(m)
        dist[:, k] = best
        sub_idx[:, k] = best_m
    return dist, sub_idx


def build_support_memory(Z_train, y_train, per_class=KNN_MEM_PER_CLASS_DIAG, seed=SEED):
    rng = np.random.RandomState(seed)
    keep = []
    for k in np.unique(y_train):
        idx = np.where(y_train == k)[0]
        rng.shuffle(idx)
        keep.append(idx[: min(len(idx), per_class)])
    keep = np.concatenate(keep, axis=0)
    Zm = normalize_rows_np(Z_train[keep])
    ym = y_train[keep].astype(np.int64)
    return Zm, ym


def local_support_consistency(Z, y_hat, Zm, ym, topk=SUPPORT_KNN, batch=512):
    Zq = normalize_rows_np(Z)
    out = np.zeros((len(Zq),), dtype=np.float32)
    for st in range(0, len(Zq), batch):
        ed = min(len(Zq), st + batch)
        sim = Zq[st:ed] @ Zm.T
        k_eff = min(topk, sim.shape[1])
        idx = np.argpartition(-sim, kth=k_eff - 1, axis=1)[:, :k_eff]
        top_sim = np.take_along_axis(sim, idx, axis=1)
        top_lab = ym[idx]
        yb = y_hat[st:ed].reshape(-1, 1)
        same_frac = (top_lab == yb).mean(axis=1).astype(np.float32)
        compact = ((np.mean(top_sim, axis=1) + 1.0) / 2.0).astype(np.float32)
        out[st:ed] = np.clip(same_frac * compact, 0.0, 1.0)
    return out.astype(np.float32)


def raw_support_region_scores(Z, support_model, Zm, ym):
    dist, sub_idx = class_distance_matrix(Z, support_model)
    order = np.argsort(dist, axis=1)
    y_hat = order[:, 0].astype(np.int64)
    y_second = order[:, 1].astype(np.int64) if dist.shape[1] > 1 else np.full((len(Z),), -1, dtype=np.int64)
    d1 = dist[np.arange(len(Z)), y_hat].astype(np.float32)
    d2 = dist[np.arange(len(Z)), y_second].astype(np.float32) if dist.shape[1] > 1 else (d1 + 1.0).astype(np.float32)
    margin = (d2 - d1).astype(np.float32)
    sub_hat = sub_idx[np.arange(len(Z)), y_hat].astype(np.int64)
    s_loc = local_support_consistency(Z, y_hat, Zm, ym, topk=SUPPORT_KNN)
    return {
        "dist_mat": dist.astype(np.float32),
        "y_hat": y_hat,
        "y_second": y_second,
        "sub_hat": sub_hat,
        "d1": d1,
        "d2": d2,
        "margin": margin,
        "s_loc": s_loc,
    }


def _quantile_or_fallback(x, q, fallback):
    x = np.asarray(x, dtype=np.float32)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return float(fallback)
    return float(np.quantile(x, q))


def calibrate_support_thresholds(raw_tr, y_tr, raw_ref, y_ref, K):
    core = np.zeros((K,), dtype=np.float32)
    outer = np.zeros((K,), dtype=np.float32)
    margin_thr = np.zeros((K,), dtype=np.float32)
    local_thr = np.zeros((K,), dtype=np.float32)
    for k in range(K):
        idx_ref = np.where((y_ref == k) & (raw_ref["y_hat"] == k))[0]
        idx_tr = np.where(y_tr == k)[0]
        use_ref = idx_ref if len(idx_ref) >= MIN_CLASS_CALIB else idx_tr
        src_d = raw_ref["d1"][use_ref] if use_ref is idx_ref else raw_tr["d1"][use_ref]
        src_m = raw_ref["margin"][use_ref] if use_ref is idx_ref else raw_tr["margin"][use_ref]
        src_l = raw_ref["s_loc"][use_ref] if use_ref is idx_ref else raw_tr["s_loc"][use_ref]
        core[k] = _quantile_or_fallback(src_d, CORE_Q, fallback=np.median(src_d) if len(src_d) else 1.0)
        outer[k] = _quantile_or_fallback(src_d, OUT_Q, fallback=np.max(src_d) if len(src_d) else core[k] + 1.0)
        if outer[k] <= core[k]:
            outer[k] = core[k] + max(1e-3, float(np.std(src_d)) if len(src_d) else 1.0)
        margin_thr[k] = _quantile_or_fallback(src_m, MARGIN_Q, fallback=np.median(src_m) if len(src_m) else 0.0)
        local_thr[k] = _quantile_or_fallback(src_l, LOCAL_Q, fallback=np.median(src_l) if len(src_l) else 0.25)
        local_thr[k] = float(np.clip(local_thr[k], 0.02, 0.98))
    return {
        "core": core.astype(np.float32),
        "outer": outer.astype(np.float32),
        "margin": margin_thr.astype(np.float32),
        "local": local_thr.astype(np.float32),
    }


def diagnose_support_regions(raw, thr):
    y_hat = raw["y_hat"]
    d1 = raw["d1"]
    margin = raw["margin"]
    s_loc = raw["s_loc"]
    core = thr["core"][y_hat]
    outer = thr["outer"][y_hat]
    margin_req = thr["margin"][y_hat]
    local_req = thr["local"][y_hat]

    pred = np.zeros((len(y_hat),), dtype=np.int64)
    unk = (d1 > outer) | (margin < margin_req) | (s_loc < local_req)
    pred[unk] = 2
    drift = (~unk) & (d1 > core)
    pred[drift] = 1

    rho = np.clip((d1 - core) / np.maximum(outer - core, 1e-6), 0.0, 1.0).astype(np.float32)
    unknown_risk = np.maximum.reduce([
        d1 / np.maximum(outer, 1e-6),
        margin_req / np.maximum(margin, 1e-6),
        local_req / np.maximum(s_loc, 1e-6),
    ]).astype(np.float32)
    drift_score = np.clip(rho, 0.0, 1.0).astype(np.float32)

    return {
        **raw,
        "pred_state": pred,
        "rho": rho,
        "unknown_risk": unknown_risk,
        "drift_score": drift_score,
    }


def plot_support_scatter(d1, margin, gt, out_png, max_points=20000):
    n = len(d1)
    if n > max_points:
        idx = np.random.RandomState(SEED).choice(n, size=max_points, replace=False)
    else:
        idx = np.arange(n)
    plt.figure(figsize=(6, 5))
    plt.scatter(d1[idx], margin[idx], s=3, c=gt[idx], alpha=0.5)
    plt.xlabel("nearest support distance d1 (lower = more supported)")
    plt.ylabel("class margin d2 - d1 (higher = less ambiguous)")
    plt.title("Module2 SR1 support-region diagnosis (0=stable,1=drift,2=unknown)")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()


def plot_confmat(cm, out_png, title):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    plt.xticks([0, 1, 2], ["Stable", "Drift", "Unknown"])
    plt.yticks([0, 1, 2], ["Stable", "Drift", "Unknown"])
    plt.xlabel("Pred")
    plt.ylabel("GT")
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

In [16]:
# =========================
# Build datasets
# =========================
def build_source_train(compact_dataset, KNOWN_TX, source_rxs):
    X_list, y_list, D_list = [], [], []
    rx_id_list = []
    for tx in KNOWN_TX:
        for rx in source_rxs:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]
                Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz)
                y_list.append(y)
                D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    if not X_list:
        raise ValueError("Empty source training set. Check protocol configuration.")
    X = np.concatenate(X_list, 0).astype(np.float32)
    y = np.concatenate(y_list, 0).astype(np.int64)
    D = np.concatenate(D_list, 0).astype(np.float32)
    rx_id = np.concatenate(rx_id_list, 0).astype(np.int64)
    return X, y, D, rx_id

def build_eval_set(compact_dataset, txs, rxs, days):
    X_list, D_list = [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]
                Xd = Xd[:n]
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz)
                D_list.append(Df)
    if not X_list:
        raise ValueError("Empty evaluation set. Check protocol configuration.")
    X = np.concatenate(X_list, 0).astype(np.float32)
    D = np.concatenate(D_list, 0).astype(np.float32)
    return X, D


In [17]:
# =========================
# SR1 notes
# =========================
# Module2 SR1 intentionally does not re-introduce additional score fusion.
# The diagnosis pipeline is:
#   nearest support distance  -> support-region attachment
#   class margin              -> ambiguity to similar known classes
#   local support consistency -> whether source neighbors actually support this attachment
# Stable / drift separation is then a class-conditional shell decision:
#   core region  -> stable-known
#   outer shell  -> known-drift
#   outside / unsupported / ambiguous -> unknown

In [18]:
# =========================
# Main loop
# =========================
def run_one_split(protocol_tag, rx_protocol_tag, tx_protocol_tag, split_id, KNOWN_TX, UNKNOWN_TX, source_rxs, drift_rx):
    protocol_dir = os.path.join(RUN_DIR, protocol_tag)
    split_dir = os.path.join(protocol_dir, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)

    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "protocol_tag": protocol_tag,
                "rx_protocol": rx_protocol_tag,
                "tx_protocol": tx_protocol_tag,
                "KNOWN_TX": KNOWN_TX,
                "UNKNOWN_TX": UNKNOWN_TX,
                "SOURCE_RXS": source_rxs,
                "DRIFT_RX": drift_rx,
                "series": "SR1",
            },
            f,
            indent=2,
        )

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX, source_rxs)
    K = len(KNOWN_TX)

    X_drRX, D_drRX = build_eval_set(compact_dataset, KNOWN_TX, drift_rx, TEST_DATES_RX)
    X_drDAY, D_drDAY = build_eval_set(compact_dataset, KNOWN_TX, source_rxs, TEST_DATES_DAY)

    X_u_in, D_u_in = build_eval_set(compact_dataset, UNKNOWN_TX, source_rxs, TEST_DATES_RX)
    X_u_drRX, D_u_drRX = build_eval_set(compact_dataset, UNKNOWN_TX, drift_rx, TEST_DATES_RX)
    X_u_drDAY, D_u_drDAY = build_eval_set(compact_dataset, UNKNOWN_TX, source_rxs, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000 * split_id + len(source_rxs))
    rows = []

    for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        rng = np.random.RandomState(SEED + 1000 * split_id + 100 * len(source_rxs) + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(ArrayDataset(X_src[idx_val], y_src[idx_val]), batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val = -1.0
        best_state = None
        patience = 0
        hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

        for ep in range(1, EPOCHS + 1):
            tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=opt)
            va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)
            hist["train_loss"].append(tr_loss)
            hist["train_acc"].append(tr_acc)
            hist["val_loss"].append(va_loss)
            hist["val_acc"].append(va_acc)
            if va_acc > best_val + 1e-4:
                best_val = va_acc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
        with open(os.path.join(fold_dir, "history.json"), "w", encoding="utf-8") as f:
            json.dump(hist, f, indent=2)
        model.load_state_dict(best_state)

        _, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
        logits_B, Z_B = infer_logits_embed(model, X_drRX, batch=512)
        logits_C, Z_C = infer_logits_embed(model, X_drDAY, batch=512)
        logits_D, Z_D = infer_logits_embed(model, X_u_in, batch=512)
        logits_E, Z_E = infer_logits_embed(model, X_u_drRX, batch=512)
        logits_F, Z_F = infer_logits_embed(model, X_u_drDAY, batch=512)

        support_model = fit_class_support_regions(Z_tr, y_src[idx_train], K)
        Zm, ym = build_support_memory(Z_tr, y_src[idx_train], per_class=KNN_MEM_PER_CLASS_DIAG, seed=SEED + 17 * fold)

        raw_tr = raw_support_region_scores(Z_tr, support_model, Zm, ym)
        raw_A = raw_support_region_scores(Z_A, support_model, Zm, ym)
        thr = calibrate_support_thresholds(raw_tr, y_src[idx_train], raw_A, y_src[idx_te], K)

        diag_A = diagnose_support_regions(raw_A, thr)
        diag_B = diagnose_support_regions(raw_support_region_scores(Z_B, support_model, Zm, ym), thr)
        diag_C = diagnose_support_regions(raw_support_region_scores(Z_C, support_model, Zm, ym), thr)
        diag_D = diagnose_support_regions(raw_support_region_scores(Z_D, support_model, Zm, ym), thr)
        diag_E = diagnose_support_regions(raw_support_region_scores(Z_E, support_model, Zm, ym), thr)
        diag_F = diagnose_support_regions(raw_support_region_scores(Z_F, support_model, Zm, ym), thr)

        gt = np.concatenate([
            np.zeros((len(Z_A),), dtype=np.int64),
            np.ones((len(Z_B),), dtype=np.int64),
            np.ones((len(Z_C),), dtype=np.int64),
            np.full((len(Z_D),), 2, dtype=np.int64),
            np.full((len(Z_E),), 2, dtype=np.int64),
            np.full((len(Z_F),), 2, dtype=np.int64),
        ])
        pred_all = np.concatenate([
            diag_A["pred_state"], diag_B["pred_state"], diag_C["pred_state"],
            diag_D["pred_state"], diag_E["pred_state"], diag_F["pred_state"],
        ])
        cm = confusion_matrix(gt, pred_all, labels=[0, 1, 2])
        plot_support_scatter(
            np.concatenate([diag_A["d1"], diag_B["d1"], diag_C["d1"], diag_D["d1"], diag_E["d1"], diag_F["d1"]]),
            np.concatenate([diag_A["margin"], diag_B["margin"], diag_C["margin"], diag_D["margin"], diag_E["margin"], diag_F["margin"]]),
            gt,
            os.path.join(fold_dir, "support_scatter.png"),
        )
        plot_confmat(cm, os.path.join(fold_dir, "support_confmat.png"), "3-class support-region confusion")

        unknown_preds = np.concatenate([diag_D["pred_state"], diag_E["pred_state"], diag_F["pred_state"]])
        drift_preds = np.concatenate([diag_B["pred_state"], diag_C["pred_state"]])
        unknown_risk_all = np.concatenate([diag_D["unknown_risk"], diag_E["unknown_risk"], diag_F["unknown_risk"]])
        drift_score_all = np.concatenate([diag_B["drift_score"], diag_C["drift_score"]])

        acc = float((raw_A["y_hat"] == y_src[idx_te]).mean())
        stable_acc_gate = float((diag_A["pred_state"] == 0).mean())
        drift_acc_rx = float((diag_B["pred_state"] == 1).mean())
        drift_acc_day = float((diag_C["pred_state"] == 1).mean())
        drift_acc_all = float((drift_preds == 1).mean())

        unknown_reject_in = float((diag_D["pred_state"] == 2).mean())
        unknown_reject_rx = float((diag_E["pred_state"] == 2).mean())
        unknown_reject_day = float((diag_F["pred_state"] == 2).mean())
        unknown_reject_all = float((unknown_preds == 2).mean())

        FRR_stable = float((diag_A["pred_state"] != 0).mean())
        false_drift_stable = float((diag_A["pred_state"] == 1).mean())
        false_unknown_stable = float((diag_A["pred_state"] == 2).mean())

        FAR_unknown_accept = float((unknown_preds != 2).mean())
        FAR_unknown_accept_in = float((diag_D["pred_state"] != 2).mean())
        FAR_unknown_accept_rx = float((diag_E["pred_state"] != 2).mean())
        FAR_unknown_accept_day = float((diag_F["pred_state"] != 2).mean())

        miss_drift_rx = float((diag_B["pred_state"] == 0).mean())
        miss_drift_day = float((diag_C["pred_state"] == 0).mean())
        miss_drift_all = float((np.sum(diag_B["pred_state"] == 0) + np.sum(diag_C["pred_state"] == 0)) / (len(diag_B["pred_state"]) + len(diag_C["pred_state"])))

        auc_unknown_in = roc_auc(
            np.concatenate([np.zeros((len(diag_A["unknown_risk"]),), dtype=np.int64), np.ones((len(diag_D["unknown_risk"]),), dtype=np.int64)]),
            np.concatenate([diag_A["unknown_risk"], diag_D["unknown_risk"]])
        )
        auc_unknown_all = roc_auc(
            np.concatenate([np.zeros((len(diag_A["unknown_risk"]),), dtype=np.int64), np.ones((len(unknown_risk_all),), dtype=np.int64)]),
            np.concatenate([diag_A["unknown_risk"], unknown_risk_all])
        )
        auc_drift_rx = roc_auc(
            np.concatenate([np.zeros((len(diag_A["drift_score"]),), dtype=np.int64), np.ones((len(diag_B["drift_score"]),), dtype=np.int64)]),
            np.concatenate([diag_A["drift_score"], diag_B["drift_score"]])
        )
        auc_drift_day = roc_auc(
            np.concatenate([np.zeros((len(diag_A["drift_score"]),), dtype=np.int64), np.ones((len(diag_C["drift_score"]),), dtype=np.int64)]),
            np.concatenate([diag_A["drift_score"], diag_C["drift_score"]])
        )
        auc_drift_all = roc_auc(
            np.concatenate([np.zeros((len(diag_A["drift_score"]),), dtype=np.int64), np.ones((len(drift_score_all),), dtype=np.int64)]),
            np.concatenate([diag_A["drift_score"], drift_score_all])
        )

        geometry_out = {
            "series": "SR1",
            "support_model_geometry": support_model["geometry"],
            "thresholds": {k: v.tolist() for k, v in thr.items()},
        }
        with open(os.path.join(fold_dir, "class_geometry.json"), "w", encoding="utf-8") as f:
            json.dump(geometry_out, f, indent=2)
        np.savez_compressed(
            os.path.join(fold_dir, "support_region_scores.npz"),
            A_pred=diag_A["pred_state"], B_pred=diag_B["pred_state"], C_pred=diag_C["pred_state"],
            D_pred=diag_D["pred_state"], E_pred=diag_E["pred_state"], F_pred=diag_F["pred_state"],
            A_yhat=diag_A["y_hat"], B_yhat=diag_B["y_hat"], C_yhat=diag_C["y_hat"],
            D_yhat=diag_D["y_hat"], E_yhat=diag_E["y_hat"], F_yhat=diag_F["y_hat"],
            A_d1=diag_A["d1"], B_d1=diag_B["d1"], C_d1=diag_C["d1"], D_d1=diag_D["d1"], E_d1=diag_E["d1"], F_d1=diag_F["d1"],
            A_margin=diag_A["margin"], B_margin=diag_B["margin"], C_margin=diag_C["margin"], D_margin=diag_D["margin"], E_margin=diag_E["margin"], F_margin=diag_F["margin"],
            A_sloc=diag_A["s_loc"], B_sloc=diag_B["s_loc"], C_sloc=diag_C["s_loc"], D_sloc=diag_D["s_loc"], E_sloc=diag_E["s_loc"], F_sloc=diag_F["s_loc"],
            A_rho=diag_A["rho"], B_rho=diag_B["rho"], C_rho=diag_C["rho"], D_rho=diag_D["rho"], E_rho=diag_E["rho"], F_rho=diag_F["rho"],
        )

        row = dict(
            split=split_id,
            fold=fold,
            acc=acc,
            stable_acc_gate=stable_acc_gate,
            drift_acc_rx=drift_acc_rx,
            drift_acc_day=drift_acc_day,
            drift_acc_all=drift_acc_all,
            unknown_reject_in=unknown_reject_in,
            unknown_reject_rx=unknown_reject_rx,
            unknown_reject_day=unknown_reject_day,
            unknown_reject_all=unknown_reject_all,
            auc_unknown_in=auc_unknown_in,
            auc_unknown_all=auc_unknown_all,
            auc_drift_rx=auc_drift_rx,
            auc_drift_day=auc_drift_day,
            auc_drift_all=auc_drift_all,
            FRR_stable=FRR_stable,
            false_drift_stable=false_drift_stable,
            false_unknown_stable=false_unknown_stable,
            FAR_unknown_accept=FAR_unknown_accept,
            FAR_unknown_accept_in=FAR_unknown_accept_in,
            FAR_unknown_accept_rx=FAR_unknown_accept_rx,
            FAR_unknown_accept_day=FAR_unknown_accept_day,
            miss_drift_rx=miss_drift_rx,
            miss_drift_day=miss_drift_day,
            miss_drift_all=miss_drift_all,
            d1_stable_mean=float(np.mean(diag_A["d1"])),
            d1_drift_mean=float(np.mean(np.concatenate([diag_B["d1"], diag_C["d1"]]))),
            d1_unknown_mean=float(np.mean(np.concatenate([diag_D["d1"], diag_E["d1"], diag_F["d1"]]))),
            margin_stable_mean=float(np.mean(diag_A["margin"])),
            margin_drift_mean=float(np.mean(np.concatenate([diag_B["margin"], diag_C["margin"]]))),
            margin_unknown_mean=float(np.mean(np.concatenate([diag_D["margin"], diag_E["margin"], diag_F["margin"]]))),
            local_stable_mean=float(np.mean(diag_A["s_loc"])),
            local_drift_mean=float(np.mean(np.concatenate([diag_B["s_loc"], diag_C["s_loc"]]))),
            local_unknown_mean=float(np.mean(np.concatenate([diag_D["s_loc"], diag_E["s_loc"], diag_F["s_loc"]]))),
            rho_stable_mean=float(np.mean(diag_A["rho"])),
            rho_drift_mean=float(np.mean(np.concatenate([diag_B["rho"], diag_C["rho"]]))),
            n_subproto_classes=int(np.sum(np.array([g["n_regions"] for g in support_model["geometry"]]) > 1)),
            confusion_matrix=cm.tolist(),
            protocol_tag=protocol_tag,
            rx_protocol=rx_protocol_tag,
            tx_protocol=tx_protocol_tag,
            source_rxs=list(source_rxs),
            drift_rx=list(drift_rx),
            known_tx=list(KNOWN_TX),
            unknown_tx=list(UNKNOWN_TX),
        )
        rows.append(row)

        with open(os.path.join(fold_dir, "metrics_module2_sr1.json"), "w", encoding="utf-8") as f:
            json.dump(row, f, indent=2)

        print(
            f"[{protocol_tag} | split {split_id} fold {fold}] "
            f"acc={acc:.3f} stable={stable_acc_gate:.3f} drift={drift_acc_all:.3f} "
            f"unkReject={unknown_reject_all:.3f} FARunk={FAR_unknown_accept:.3f} "
            f"subprotoCls={row['n_subproto_classes']}"
        )

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows


all_rows = []
protocol_specs = []
for protocol_idx, ps in enumerate(PROTOCOL_SPECS, start=1):
    protocol_tag = ps["protocol_tag"]
    print("\n==============================")
    print(f"PROTOCOL {protocol_idx}/{len(PROTOCOL_SPECS)}: {protocol_tag}")
    print("SOURCE_RXS:", ps["source_rxs"], "| DRIFT_RX:", ps["drift_rx"])
    print("TX protocol:", ps["tx_protocol"])
    protocol_specs.append(ps)
    for split_pos, tx_split in enumerate(ps["tx_splits"], start=1):
        if isinstance(tx_split, dict):
            split_id = int(tx_split.get("split_id", split_pos))
            KNOWN_TX = list(tx_split["known"])
            UNKNOWN_TX = list(tx_split["unknown"])
        else:
            split_id = split_pos
            KNOWN_TX, UNKNOWN_TX = tx_split
            KNOWN_TX = list(KNOWN_TX)
            UNKNOWN_TX = list(UNKNOWN_TX)
        print("\n=== TX split", split_id, "===")
        print("KNOWN_TX:", KNOWN_TX)
        print("UNKNOWN_TX:", UNKNOWN_TX)
        all_rows.extend(run_one_split(
            protocol_tag=protocol_tag,
            rx_protocol_tag=ps["rx_protocol"],
            tx_protocol_tag=ps["tx_protocol"],
            split_id=split_id,
            KNOWN_TX=KNOWN_TX,
            UNKNOWN_TX=UNKNOWN_TX,
            source_rxs=ps["source_rxs"],
            drift_rx=ps["drift_rx"],
        ))

with open(os.path.join(RUN_DIR, "protocol_specs.json"), "w", encoding="utf-8") as f:
    json.dump(protocol_specs, f, indent=2)
with open(os.path.join(RUN_DIR, "metrics_all_rows.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)


=== RX3-9_TX2-4 ===
[split 1] KNOWN_TX=['20-15', '8-20'] UNKNOWN_TX=['14-10', '14-7', '20-19', '6-15']
[RX3-9_TX2-4 split 1 fold 1] acc=0.998 stable=0.882 driftRX=0.215 driftDAY=0.096 unkRej=0.476 FARunk=0.524 missDr=0.815 AUCu=0.864 AUCd=0.393
[RX3-9_TX2-4 split 1 fold 2] acc=0.999 stable=0.872 driftRX=0.254 driftDAY=0.086 unkRej=0.343 FARunk=0.657 missDr=0.788 AUCu=0.846 AUCd=0.402
[RX3-9_TX2-4 split 1 fold 3] acc=0.997 stable=0.865 driftRX=0.201 driftDAY=0.083 unkRej=0.566 FARunk=0.434 missDr=0.828 AUCu=0.862 AUCd=0.331
[RX3-9_TX2-4 split 1 fold 4] acc=1.000 stable=0.890 driftRX=0.255 driftDAY=0.154 unkRej=0.615 FARunk=0.385 missDr=0.770 AUCu=0.875 AUCd=0.404
[RX3-9_TX2-4 split 1 fold 5] acc=0.993 stable=0.881 driftRX=0.219 driftDAY=0.104 unkRej=0.392 FARunk=0.608 missDr=0.810 AUCu=0.828 AUCd=0.435
[split 2] KNOWN_TX=['14-7', '6-15'] UNKNOWN_TX=['14-10', '20-15', '20-19', '8-20']
[RX3-9_TX2-4 split 2 fold 1] acc=0.999 stable=0.868 driftRX=0.102 driftDAY=0.023 unkRej=0.569 FARunk=0.

In [19]:
# =========================
# Summary
# =========================
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

METRIC_KEYS = [
    "acc",
    "stable_acc_gate",
    "drift_acc_rx",
    "drift_acc_day",
    "drift_acc_all",
    "unknown_reject_in",
    "unknown_reject_rx",
    "unknown_reject_day",
    "unknown_reject_all",
    "auc_unknown_in",
    "auc_unknown_all",
    "auc_drift_rx",
    "auc_drift_day",
    "auc_drift_all",
    "FRR_stable",
    "false_drift_stable",
    "false_unknown_stable",
    "FAR_unknown_accept",
    "FAR_unknown_accept_in",
    "FAR_unknown_accept_rx",
    "FAR_unknown_accept_day",
    "miss_drift_rx",
    "miss_drift_day",
    "miss_drift_all",
    "d1_stable_mean",
    "d1_drift_mean",
    "d1_unknown_mean",
    "margin_stable_mean",
    "margin_drift_mean",
    "margin_unknown_mean",
    "local_stable_mean",
    "local_drift_mean",
    "local_unknown_mean",
    "rho_stable_mean",
    "rho_drift_mean",
    "n_subproto_classes",
]

def summarize_rows(rows, metric_keys=METRIC_KEYS):
    summary = {"n_rows": int(len(rows))}
    for key in metric_keys:
        m, s = mean_std([r[key] for r in rows])
        summary[key] = {"mean": m, "std": s}
    cm_sum = np.zeros((3, 3), dtype=np.int64)
    for r in rows:
        cm_sum += np.array(r["confusion_matrix"], dtype=np.int64)
    summary["confusion_matrix_sum"] = cm_sum.tolist()
    return summary


def summarize_by_group(rows, group_key, metric_keys=METRIC_KEYS):
    groups = {}
    for r in rows:
        groups.setdefault(r[group_key], []).append(r)
    out = {}
    for key, group_rows in groups.items():
        out[key] = summarize_rows(group_rows, metric_keys=metric_keys)
    return out

summary = summarize_rows(all_rows, metric_keys=METRIC_KEYS)
summary_by_protocol = summarize_by_group(all_rows, "protocol_tag", metric_keys=METRIC_KEYS)
summary_by_rx_protocol = summarize_by_group(all_rows, "rx_protocol", metric_keys=METRIC_KEYS)
summary_by_tx_protocol = summarize_by_group(all_rows, "tx_protocol", metric_keys=METRIC_KEYS)

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
with open(os.path.join(RUN_DIR, "summary_by_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(summary_by_protocol, f, indent=2)
with open(os.path.join(RUN_DIR, "summary_by_rx_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(summary_by_rx_protocol, f, indent=2)
with open(os.path.join(RUN_DIR, "summary_by_tx_protocol.json"), "w", encoding="utf-8") as f:
    json.dump(summary_by_tx_protocol, f, indent=2)

print("=== Overall mean ± std over protocol × split × fold ===")
for k, v in summary.items():
    if k in ["n_rows", "confusion_matrix_sum"]:
        continue
    print(f"{k:28s}: {v['mean']:.3f} ± {v['std']:.3f}")

print("\nConfusion matrix SUM (rows=GT, cols=Pred) [Stable, Drift, Unknown]:")
print(np.array(summary["confusion_matrix_sum"], dtype=np.int64))

print("\n=== Per-protocol quick view ===")
for protocol_tag in sorted(summary_by_protocol.keys()):
    s = summary_by_protocol[protocol_tag]
    print(
        f"[{protocol_tag}] "
        f"FARunk={s['FAR_unknown_accept']['mean']:.3f} ± {s['FAR_unknown_accept']['std']:.3f} | "
        f"missRX={s['miss_drift_rx']['mean']:.3f} ± {s['miss_drift_rx']['std']:.3f} | "
        f"missDAY={s['miss_drift_day']['mean']:.3f} ± {s['miss_drift_day']['std']:.3f} | "
        f"AUCuAll={s['auc_unknown_all']['mean']:.3f} ± {s['auc_unknown_all']['std']:.3f} | "
        f"AUCdAll={s['auc_drift_all']['mean']:.3f} ± {s['auc_drift_all']['std']:.3f} | "
        f"subProto={s['n_subproto_classes']['mean']:.2f}"
    )

print("\nSaved overall summary to:", os.path.join(RUN_DIR, "summary_mean_std.json"))
print("Saved per-protocol summary to:", os.path.join(RUN_DIR, "summary_by_protocol.json"))
print("Saved RX-family summary to:", os.path.join(RUN_DIR, "summary_by_rx_protocol.json"))
print("Saved TX-family summary to:", os.path.join(RUN_DIR, "summary_by_tx_protocol.json"))

=== Overall mean ± std over protocol × split × fold ===
acc                         : 0.996 ± 0.003
stable_acc_gate             : 0.857 ± 0.051
drift_acc_rx                : 0.146 ± 0.132
drift_acc_day               : 0.092 ± 0.068
drift_acc_all               : 0.122 ± 0.088
unknown_reject_in           : 0.640 ± 0.228
unknown_reject_rx           : 0.673 ± 0.235
unknown_reject_day          : 0.638 ± 0.216
unknown_reject_all          : 0.656 ± 0.215
auc_unknown_in              : 0.838 ± 0.105
auc_unknown_all             : 0.845 ± 0.100
auc_drift_rx                : 0.414 ± 0.155
auc_drift_day               : 0.484 ± 0.105
auc_drift_all               : 0.443 ± 0.125
FRR_stable                  : 0.143 ± 0.051
false_drift_stable          : 0.041 ± 0.007
false_unknown_stable        : 0.102 ± 0.055
FAR_unknown_accept          : 0.344 ± 0.215
FAR_unknown_accept_in       : 0.360 ± 0.228
FAR_unknown_accept_rx       : 0.327 ± 0.235
FAR_unknown_accept_day      : 0.362 ± 0.216
miss_drift_rx       